# Importing all modules we need

In [ ]:
import math

import pandas as pd
import os

# Loading all files we will need later on


In [ ]:
path_fitts = os.path.join('results', 'fitts')
path_steering = os.path.join('results', 'steering')

files_fitts = [file for file in os.listdir(path_fitts) if file.endswith('.csv')]
files_steering = [file for file in os.listdir(path_steering) if file.endswith('.csv')]

dfs_fitts = []

for file in files_fitts:
    path = os.path.join(path_fitts, file)
    dfs_fitts.append(pd.read_csv(path))

df_fitts = pd.concat(dfs_fitts, ignore_index=True)

dfs_steering = []

for file in files_steering:
    path = os.path.join(path_steering, file)
    dfs_steering.append(pd.read_csv(path))

df_steering = pd.concat(dfs_steering, ignore_index=True)

In [ ]:
print(df_fitts)

In [ ]:
df_steering

In [ ]:
fitts_runs_df = (
    df_fitts
    .groupby(
        [
            "part_id",
            "input_method",
            "delay",
            "radius",
            "distance",
            "iteration",
        ],
        as_index=False,
    )
    .agg(
        start_time=("timestamp", "min"),
        end_time=("timestamp", "max"),
        num_clicks=("hit", "count"),
        num_hits = ("hit", "sum"),
        num_targets=("target_id", "count"),
    )
)

fitts_runs_df["duration_ms"] = (
    fitts_runs_df["end_time"] - fitts_runs_df["start_time"]
)

fitts_runs_df['accuracy'] = (
    fitts_runs_df['num_hits'] / fitts_runs_df['num_targets']
)

print(fitts_runs_df)

In [ ]:
mean_df_fitts = (
    fitts_runs_df.groupby(
        ['part_id', 'input_method', 'delay', 'radius', 'distance'], as_index=False
    ).agg(
        mean_duration=("duration_ms", "mean"),
        mean_accuracy=("accuracy", "mean"),
    )
)

mean_df_fitts['mean_duration_seconds'] = (mean_df_fitts['mean_duration'] / 1000).round(2)
mean_df_fitts['no_errors'] = mean_df_fitts['mean_accuracy'] == 1

In [ ]:
print(mean_df_fitts)

In [ ]:
mean_fitts_combined = mean_df_fitts.groupby(
    ['part_id', 'input_method', 'delay'], as_index=False
).agg(
    total_mean_duration=("mean_duration_seconds", "mean"),
    total_mean_accuracy=("mean_accuracy", "mean"),
    total_no_errors=("no_errors", "sum")
)

print(mean_fitts_combined)

In [ ]:
mean_fitts_final = mean_fitts_combined.groupby(
    ['input_method', 'delay'], as_index=False
).agg(
    mean_duration=("total_mean_duration", "mean"),
    mean_accuracy=("total_mean_accuracy", "mean"),
)
mean_fitts_final['mean_duration'] = mean_fitts_final['mean_duration'].round(2)
print(mean_fitts_final)

In [ ]:
import matplotlib.pyplot as plt

plot_data = {
    "Mouse": mean_df_fitts[
        (mean_df_fitts["input_method"] == "mouse") &
        (mean_df_fitts["delay"] == 0)
    ]["mean_duration_seconds"],

    "Mouse + delay": mean_df_fitts[
        (mean_df_fitts["input_method"] == "mouse") &
        (mean_df_fitts["delay"] > 0)
    ]["mean_duration_seconds"],

    "Hand": mean_df_fitts[
        mean_df_fitts["input_method"] == "pose"
    ]["mean_duration_seconds"],

    "Touchpad": mean_df_fitts[
        mean_df_fitts["input_method"] == "touchpad"
    ]["mean_duration_seconds"],
}

plt.figure(figsize=(8, 5))
plt.boxplot(plot_data.values(), tick_labels=plot_data.keys())
plt.title("Completion Time by Input Method")
plt.xlabel("Input Method")
plt.ylabel("Mean Completion Time (s)")
plt.grid(axis="y")
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

plot_data = {
    "Mouse": mean_df_fitts[
        (mean_df_fitts["input_method"] == "mouse") &
        (mean_df_fitts["delay"] == 0)
    ]["mean_accuracy"],

    "Mouse + delay": mean_df_fitts[
        (mean_df_fitts["input_method"] == "mouse") &
        (mean_df_fitts["delay"] > 0)
    ]["mean_accuracy"],

    "Hand": mean_df_fitts[
        mean_df_fitts["input_method"] == "pose"
    ]["mean_accuracy"],

    "Touchpad": mean_df_fitts[
        mean_df_fitts["input_method"] == "touchpad"
    ]["mean_accuracy"],
}

plt.figure(figsize=(8, 5))
plt.boxplot(plot_data.values(), tick_labels=plot_data.keys())
plt.title("Completion Time by Input Method")
plt.xlabel("Input Method")
plt.ylabel("Mean Accuracy")
plt.grid(axis="y")
plt.tight_layout()
plt.show()

In [ ]:
print(df_steering)

In [ ]:
df_steering['duration'] = ((df_steering['end_time'] - df_steering['start_time']) / 1000).round(2)

df_steering_iterations = df_steering.groupby(
    ['part_id', 'input_method', 'delay', 'width', 'distance'], as_index=False
).agg(
    mean_duration=("duration", "mean"),
    total_errors=('errors', 'sum')
)

df_steering_iterations['error_rate'] = df_steering_iterations['total_errors'] / 3


In [ ]:
print(df_steering_iterations)

In [ ]:
df_steering_combined = df_steering_iterations.groupby(
    ['input_method', 'delay'], as_index=False
).agg(
    total_errors=("total_errors", "sum"),
    mean_error_rate=("error_rate", "mean"),
    mean_duration=('mean_duration', 'mean')
)

In [ ]:
print(df_steering_combined)

In [ ]:
df_steering_participants = df_steering_iterations.groupby(
    ['part_id', 'input_method', 'delay'], as_index=False
).agg(
    total_errors=("total_errors", "sum"),
    mean_error_rate=("error_rate", "mean"),
    mean_duration=('mean_duration', 'mean')
)

In [ ]:
print(df_steering_participants)

In [ ]:
import matplotlib.pyplot as plt

plot_data = {
    "Mouse": df_steering_participants[
        (df_steering_participants["input_method"] == "mouse") &
        (df_steering_participants["delay"] == 0)
    ]["mean_duration"],

    "Mouse + delay": df_steering_participants[
        (df_steering_participants["input_method"] == "mouse") &
        (df_steering_participants["delay"] > 0)
    ]["mean_duration"],

    "Pose": df_steering_participants[
        df_steering_participants["input_method"] == "pose"
    ]["mean_duration"],

    "Touchpad": df_steering_participants[
        df_steering_participants["input_method"] == "touchpad"
    ]["mean_duration"],
}

plt.figure(figsize=(8, 5))
plt.boxplot(plot_data.values(), tick_labels=plot_data.keys())

plt.title("Steering Task: Completion Time")
plt.xlabel("Input Method")
plt.ylabel("Mean Completion Time (s)")
plt.grid(axis="y")

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

plot_data = {
    "Mouse": df_steering_participants[
        (df_steering_participants["input_method"] == "mouse") &
        (df_steering_participants["delay"] == 0)
    ]["mean_error_rate"],

    "Mouse + delay": df_steering_participants[
        (df_steering_participants["input_method"] == "mouse") &
        (df_steering_participants["delay"] > 0)
    ]["mean_error_rate"],

    "Pose": df_steering_participants[
        df_steering_participants["input_method"] == "pose"
    ]["mean_error_rate"],

    "Touchpad": df_steering_participants[
        df_steering_participants["input_method"] == "touchpad"
    ]["mean_error_rate"],
}

plt.figure(figsize=(8, 5))
plt.boxplot(plot_data.values(), tick_labels=plot_data.keys())

plt.title("Steering Task: Error Rate")
plt.xlabel("Input Method")
plt.ylabel("Mean Error Rate")
plt.ylim(0, 1)
plt.grid(axis="y")

plt.tight_layout()
plt.show()